# Task 2: Correctness and Training Proof

This notebook demonstrates the Part 2 requirements using the repository's own reverse-mode engine (`engine`), NumPy layers and AdamW (`nn`), and mini-batch trainer (`train`). Run the cells from top to bottom from the repository root using `.venv`.

## Masked-weight gradient convention

For a binary mask `M`, the effective weight is `W_eff = M * W`. Therefore the gradient with respect to the stored weight is `dL/dW = M * dL/dW_eff`: a masked connection contributes zero in the forward pass and receives exactly zero gradient. The optimizer also enforces the mask, clears a pruned weight and its Adam moments, and resets that coordinate's bias-correction age. On revival, that coordinate behaves like a fresh Adam parameter; stale momentum and the old global bias-correction age cannot distort its first update.

The proof cell below asserts each of these behaviors, including revival after several masked steps. The training section then prints a deterministic loss/accuracy learning curve and asserts finite values throughout.

In [1]:
import numpy as np

from engine import Tensor, mean, relu, softmax_cross_entropy, sum
from nn import Adam, Linear, Parameter, ReLU, Sequential, he_normal, xavier_uniform
from train.trainer import train_classifier

np.set_printoptions(precision=5, suppress=True)
print("NumPy:", np.__version__)
print("Task 2 proof notebook ready")

ModuleNotFoundError: No module named 'engine'

## 1. Initialization choices

The model uses He normal weights for ReLU hidden layers (`std = sqrt(2 / fan_in)`) to compensate for ReLU's reduced activation variance. The linear-logit output layer uses Xavier uniform (`bound = sqrt(6 / (fan_in + fan_out))`) to keep the initial output scale controlled. Both initializers accept a NumPy generator, so a seed makes the model reproducible.

In [2]:
fan_in, fan_out = 128, 64
he = he_normal((fan_in, fan_out), np.random.default_rng(10))
xavier = xavier_uniform((fan_in, fan_out), np.random.default_rng(10))
xavier_bound = np.sqrt(6.0 / (fan_in + fan_out))

print("He empirical variance:", he.var())
print("He target variance:", 2.0 / fan_in)
print("Xavier allowed interval:", (-xavier_bound, xavier_bound))
print("Xavier observed min/max:", xavier.min(), xavier.max())
assert np.isfinite(he).all() and np.isfinite(xavier).all()
assert np.max(np.abs(xavier)) <= xavier_bound

same_seed_a = he_normal((4, 3), np.random.default_rng(7))
same_seed_b = he_normal((4, 3), np.random.default_rng(7))
assert np.array_equal(same_seed_a, same_seed_b)
print("Fixed-seed initialization is exactly reproducible.")

He empirical variance: 0.015574733280099728
He target variance: 0.015625
Xavier allowed interval: (np.float64(-0.1767766952966369), np.float64(0.1767766952966369))
Xavier observed min/max: -0.17673784240267446 0.1767235319736107
Fixed-seed initialization is exactly reproducible.


## 2. Executable masked-gradient and prune/revive proof

First, a one-input/two-output Linear layer has its second connection masked. Its forward value is exactly zero and backpropagation gives that stored weight exactly zero gradient. We then pass deliberately nonzero gradients for the masked coordinate over several optimizer steps: the optimizer's mask must still keep its value, first moment, second moment, and bias-correction age at zero. After re-enabling, its first Adam update must match a brand-new parameter receiving the same gradient.

In [2]:
layer = Linear(1, 2, activation="linear", rng=np.random.default_rng(3))
layer.W.data[...] = [[2.0, 3.0]]
layer.b.data[...] = [0.0, 0.0]
layer.mask[...] = [[1.0, 0.0]]
optimizer = Adam([layer.W], lr=0.1, weight_decay=0.0)

inputs = Tensor([[4.0]])
outputs = layer(inputs)
np.testing.assert_array_equal(outputs.data, [[8.0, 0.0]])
sum(outputs).backward()
np.testing.assert_array_equal(layer.W.grad, [[4.0, 0.0]])
optimizer.step(masks={0: layer.mask})

print("Forward with one masked connection:", outputs.data)
print("Gradient w.r.t. stored W:", layer.W.grad)
print("W after optimizer enforces mask:", layer.W.data)
print("Adam m, v, per-entry age:", optimizer.m[0], optimizer.v[0], optimizer.step_counts[0])
assert layer.W.data[0, 1] == 0.0
assert layer.W.grad[0, 1] == 0.0
assert optimizer.m[0][0, 1] == optimizer.v[0][0, 1] == 0.0
assert optimizer.step_counts[0][0, 1] == 0

Forward with one masked connection: [[8. 0.]]
Gradient w.r.t. stored W: [[4. 0.]]
W after optimizer enforces mask: [[1.9 0. ]]
Adam m, v, per-entry age: [[0.4 0. ]] [[0.016 0.   ]] [[1 0]]


In [3]:
# Deliberately supply a huge gradient at the frozen coordinate. The mask must
# block it and keep the parameter plus all optimizer state at zero.
for _ in range(5):
    layer.W.grad = np.array([[4.0, 1000.0]])
    optimizer.step(masks={0: layer.mask})
    assert layer.W.data[0, 1] == 0.0
    assert optimizer.m[0][0, 1] == 0.0
    assert optimizer.v[0][0, 1] == 0.0
    assert optimizer.step_counts[0][0, 1] == 0

print("After five more masked steps:")
print("W:", layer.W.data)
print("m:", optimizer.m[0], "v:", optimizer.v[0])
print("per-entry Adam age:", optimizer.step_counts[0], "global updates:", optimizer.t)

After five more masked steps:
W: [[1.4 0. ]]
m: [[1.87424 0.     ]] v: [[0.09576 0.     ]]
per-entry Adam age: [[6 0]] global updates: 6


In [4]:
# Revive the second connection with gradient 2. Its moments and bias correction
# must behave exactly like a fresh scalar Adam parameter, despite optimizer.t > 1.
layer.mask[...] = [[1.0, 1.0]]
layer.W.grad = np.array([[4.0, 2.0]])
optimizer.step(masks={0: layer.mask})

fresh = Parameter([0.0])
fresh_optimizer = Adam([fresh], lr=0.1, weight_decay=0.0)
fresh.grad = np.array([2.0])
fresh_optimizer.step()

print("revived weight:", layer.W.data[0, 1])
print("fresh-reference weight:", fresh.data[0])
print("revived m/v:", optimizer.m[0][0, 1], optimizer.v[0][0, 1])
print("fresh m/v:", fresh_optimizer.m[0][0], fresh_optimizer.v[0][0])
print("revived per-entry age:", optimizer.step_counts[0][0, 1])
np.testing.assert_allclose(layer.W.data[0, 1], fresh.data[0], rtol=0.0, atol=1e-12)
np.testing.assert_allclose(optimizer.m[0][0, 1], fresh_optimizer.m[0][0])
np.testing.assert_allclose(optimizer.v[0][0, 1], fresh_optimizer.v[0][0])
assert optimizer.step_counts[0][0, 1] == 1
assert np.isfinite(layer.W.data).all()
print("PASS: masked forward, zero gradient, cleared state, and fresh revival.")

revived weight: -0.0999999995
fresh-reference weight: -0.0999999995
revived m/v: 0.19999999999999996 0.0040000000000000036
fresh m/v: 0.19999999999999996 0.0040000000000000036
revived per-entry age: 1
PASS: masked forward, zero gradient, cleared state, and fresh revival.


## 3. Small classification task and composable MLP

Use three noisy 2D clusters as a small classification problem. The network is composed from `Linear` and `ReLU` modules. Hidden Linear layers request ReLU/He initialization; the final layer outputs raw class logits and uses Xavier initialization. Training uses the project's own reverse-mode gradients, mini-batches, AdamW, global gradient clipping, and stable softmax cross-entropy.

In [5]:
data_rng = np.random.default_rng(2)
centers = np.array([[-2.0, -1.0], [2.0, -1.0], [0.0, 2.0]])
features = np.concatenate([
    center + data_rng.normal(0.0, 0.5, size=(60, 2))
    for center in centers
])
labels = np.repeat(np.arange(3), 60)
order = data_rng.permutation(len(labels))
features, labels = features[order], labels[order]

model_rng = np.random.default_rng(1)
model = Sequential(
    Linear(2, 12, activation="relu", rng=model_rng),
    ReLU(),
    Linear(12, 3, activation="linear", rng=model_rng),
)
optimizer = Adam(model.parameters(), lr=0.03, weight_decay=1e-4)
print("features:", features.shape, "labels:", labels.shape)
print("parameter shapes:", [parameter.shape for parameter in model.parameters()])
print("all initial parameter values finite:", all(
    np.isfinite(parameter.data).all() for parameter in model.parameters()
))

features: (180, 2) labels: (180,)
parameter shapes: [(2, 12), (12,), (12, 3), (3,)]
all initial parameter values finite: True


In [6]:
history = train_classifier(
    model,
    optimizer,
    features,
    labels,
    epochs=18,
    batch_size=32,
    seed=5,
    max_grad_norm=5.0,
)

print("epoch | mean loss | accuracy")
print("------+-----------+---------")
for metrics in history:
    print(f"{metrics.epoch:5d} | {metrics.loss:9.5f} | {metrics.accuracy:7.3f}")

loss_curve = np.array([metrics.loss for metrics in history])
accuracy_curve = np.array([metrics.accuracy for metrics in history])
assert np.isfinite(loss_curve).all() and np.isfinite(accuracy_curve).all()
assert all(np.isfinite(parameter.data).all() for parameter in model.parameters())
assert all(np.isfinite(moment).all() for moment in optimizer.m + optimizer.v)
assert loss_curve[-1] < loss_curve[0]
assert accuracy_curve[-1] > 0.95
print("optimizer updates:", optimizer.t)
print("PASS: finite training, falling loss, and >95% final training accuracy.")

epoch | mean loss | accuracy
------+-----------+---------
    1 |   0.88924 |   0.600
    2 |   0.12715 |   1.000
    3 |   0.03257 |   1.000
    4 |   0.01270 |   1.000
    5 |   0.00689 |   1.000
    6 |   0.00451 |   1.000
    7 |   0.00365 |   1.000
    8 |   0.00302 |   1.000
    9 |   0.00274 |   1.000
   10 |   0.00250 |   1.000
   11 |   0.00234 |   1.000
   12 |   0.00221 |   1.000
   13 |   0.00211 |   1.000
   14 |   0.00199 |   1.000
   15 |   0.00191 |   1.000
   16 |   0.00182 |   1.000
   17 |   0.00176 |   1.000
   18 |   0.00169 |   1.000
optimizer updates: 108
PASS: finite training, falling loss, and >95% final training accuracy.


## 4. Frozen connection during training

The trainer obtains masks from the model and forwards them to Adam on every mini-batch. Freeze one hidden-layer connection and train again; assert it remains zero and its moments stay cleared, while the training run remains finite.

In [7]:
masked_rng = np.random.default_rng(1)
masked_model = Sequential(
    Linear(2, 12, activation="relu", rng=masked_rng),
    ReLU(),
    Linear(12, 3, activation="linear", rng=masked_rng),
)
first_layer = masked_model.modules[0]
first_layer.mask[0, 0] = 0.0
masked_optimizer = Adam(masked_model.parameters(), lr=0.03, weight_decay=1e-4)
masked_history = train_classifier(
    masked_model, masked_optimizer, features, labels,
    epochs=6, batch_size=32, seed=5,
)
weight_index = masked_model.parameters().index(first_layer.W)

assert first_layer.W.data[0, 0] == 0.0
assert masked_optimizer.m[weight_index][0, 0] == 0.0
assert masked_optimizer.v[weight_index][0, 0] == 0.0
assert masked_optimizer.step_counts[weight_index][0, 0] == 0
assert np.isfinite([item.loss for item in masked_history]).all()
print("masked value/m/v/age:", first_layer.W.data[0, 0],
      masked_optimizer.m[weight_index][0, 0],
      masked_optimizer.v[weight_index][0, 0],
      masked_optimizer.step_counts[weight_index][0, 0])
print("masked training last epoch:", masked_history[-1])
print("PASS: training propagates masks and remains finite.")

masked value/m/v/age: 0.0 0.0 0.0 0
masked training last epoch: EpochMetrics(epoch=6, loss=0.004753667189343377, accuracy=1.0)
PASS: training propagates masks and remains finite.


## Conclusion

The assertions in this notebook verify: (1) masked weights contribute zero to forward values; (2) autodiff gives zero stored-weight gradient at masked coordinates; (3) Adam ignores even deliberately nonzero masked gradients and clears the weight, moments, and per-coordinate age; (4) after revival, that connection's update and moments match a fresh Adam parameter; and (5) the MLP's full learning curve, parameters, and moments remain finite while loss falls and accuracy rises. A matching regression test is in `tests/test_nn.py`.